# Imports and data loading

In [1]:
import os
os.environ['HF_HOME'] = '/scratch/bhanushali.ar/netflix_semantic/hf_cache'

import ast
import json
import random
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv(dotenv_path='/scratch/bhanushali.ar/netflix_semantic/.env')

df = pd.read_csv('/scratch/bhanushali.ar/netflix_semantic/data/processed/movies_processed.csv')
print(df.shape, torch.cuda.get_device_name(0))

(4391, 11) NVIDIA H200


In [2]:
import json
with open('/scratch/bhanushali.ar/netflix_semantic/data/processed/similar_map.json') as f:
    similar_map = {int(k): v for k, v in json.load(f).items()}

print(f"loaded similar movies for {len(similar_map)} movies")

loaded similar movies for 4391 movies


# Training pairs

In [3]:
random.seed(42)

id_to_text = dict(zip(df['id'], df['rich_text']))
all_ids = set(df['id'].tolist())

positive_pairs = []
for movie_id, similar_ids in similar_map.items():
    for sim_id in similar_ids:
        if sim_id in id_to_text:
            positive_pairs.append((id_to_text[movie_id], id_to_text[sim_id]))

negative_pairs = []
id_list = list(all_ids)
while len(negative_pairs) < len(positive_pairs):
    a, b = random.sample(id_list, 2)
    if b not in similar_map.get(a, []):
        negative_pairs.append((id_to_text[a], id_to_text[b]))

print(f"positive pairs: {len(positive_pairs)}, negative pairs: {len(negative_pairs)}")

positive pairs: 20867, negative pairs: 20867


# QLoRA

In [4]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "NousResearch/Meta-Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModel.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/home/bhanushali.ar/.conda/envs/netflix-semantic/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   0%|          | 1/290 [00:40<3:17:07, 40.93s/it]/home/bhanushali.ar/.conda/envs/netflix-semantic/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 290/290 [07:00<00:00,  1.45s/it]
[transformers] LlamaModel LOAD REPORT from: NousResearch/Meta-Llama-3.1-8B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

trainable params: 6,815,744 || all params: 7,511,740,416 || trainable%: 0.0907


# Training

In [5]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class MoviePairDataset(Dataset):
    def __init__(self, positive_pairs, negative_pairs, tokenizer, max_length=128):
        self.pairs = [(a, b, 1.0) for a, b in positive_pairs] + [(a, b, 0.0) for a, b in negative_pairs]
        random.shuffle(self.pairs)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        text_a, text_b, label = self.pairs[idx]
        enc_a = self.tokenizer(text_a, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        enc_b = self.tokenizer(text_b, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        return {
            'input_ids_a': enc_a['input_ids'].squeeze(),
            'attention_mask_a': enc_a['attention_mask'].squeeze(),
            'input_ids_b': enc_b['input_ids'].squeeze(),
            'attention_mask_b': enc_b['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.float16),
        }

def mean_pooling(output, attention_mask):
    token_embeddings = output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).to(token_embeddings.dtype)
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

all_pairs = list(zip(positive_pairs, [1.0]*len(positive_pairs))) + list(zip(negative_pairs, [0.0]*len(negative_pairs)))
random.shuffle(all_pairs)

split = int(len(all_pairs) * 0.9)
train_data = all_pairs[:split]
val_data = all_pairs[split:]

train_pos = [(a, b) for (a, b), label in train_data if label == 1.0]
train_neg = [(a, b) for (a, b), label in train_data if label == 0.0]
val_pos = [(a, b) for (a, b), label in val_data if label == 1.0]
val_neg = [(a, b) for (a, b), label in val_data if label == 0.0]

train_dataset = MoviePairDataset(train_pos, train_neg, tokenizer)
val_dataset = MoviePairDataset(val_pos, val_neg, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"train: {len(train_dataset)}, val: {len(val_dataset)}")

train: 37560, val: 4174


In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
accumulation_steps = 4
epochs = 3
SAVE_PATH = '/scratch/bhanushali.ar/netflix_semantic/models/finetuned'

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        with torch.amp.autocast('cuda'):
            out_a = model(input_ids=batch['input_ids_a'].cuda(), attention_mask=batch['attention_mask_a'].cuda())
            out_b = model(input_ids=batch['input_ids_b'].cuda(), attention_mask=batch['attention_mask_b'].cuda())

            emb_a = mean_pooling(out_a, batch['attention_mask_a'].cuda())
            emb_b = mean_pooling(out_b, batch['attention_mask_b'].cuda())

            cos_sim = F.cosine_similarity(emb_a, emb_b)
            labels = batch['label'].cuda()
            loss = F.mse_loss(cos_sim, labels) / accumulation_steps

        loss.backward()
        total_loss += loss.item() * accumulation_steps

        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if (step + 1) % 500 == 0:
            print(f"epoch {epoch+1} step {step+1}/{len(train_loader)} loss: {total_loss/(step+1):.4f}")

    avg_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            with torch.amp.autocast('cuda'):
                out_a = model(input_ids=batch['input_ids_a'].cuda(), attention_mask=batch['attention_mask_a'].cuda())
                out_b = model(input_ids=batch['input_ids_b'].cuda(), attention_mask=batch['attention_mask_b'].cuda())
                emb_a = mean_pooling(out_a, batch['attention_mask_a'].cuda())
                emb_b = mean_pooling(out_b, batch['attention_mask_b'].cuda())
                cos_sim = F.cosine_similarity(emb_a, emb_b)
                labels = batch['label'].cuda()
                val_loss += F.mse_loss(cos_sim, labels).item()

    val_loss /= len(val_loader)
    print(f"Epoch {epoch+1}/{epochs} — train_loss: {avg_loss:.4f}, val_loss: {val_loss:.4f}")

    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    print(f"checkpoint saved after epoch {epoch+1}")

epoch 1 step 500/2348 loss: 0.2022
epoch 1 step 1000/2348 loss: 0.1918
epoch 1 step 1500/2348 loss: 0.1888
epoch 1 step 2000/2348 loss: 0.1868
Epoch 1/3 — train_loss: 0.1855, val_loss: 0.1791
checkpoint saved after epoch 1
epoch 2 step 500/2348 loss: 0.1672
epoch 2 step 1000/2348 loss: 0.1675
epoch 2 step 1500/2348 loss: 0.1684
epoch 2 step 2000/2348 loss: 0.1697
Epoch 2/3 — train_loss: 0.1700, val_loss: 0.1830
checkpoint saved after epoch 2
epoch 3 step 500/2348 loss: 0.1596
epoch 3 step 1000/2348 loss: 0.1637
epoch 3 step 1500/2348 loss: 0.1644
epoch 3 step 2000/2348 loss: 0.1635
Epoch 3/3 — train_loss: 0.1638, val_loss: 0.1728
checkpoint saved after epoch 3
